# Ollama Webpage Summarizer Challenge

Completed Week 1 Day 2 homework: adapt the Day 1 webpage summarizer so it uses a local open-source model through Ollama instead of OpenAI's paid API.

## Prerequisites

Before running the notebook, make sure Ollama is installed and running:

```bash
ollama serve
ollama pull llama3.2
```

If your machine is smaller, use `llama3.2:1b` and update the `MODEL` variable below.

In [ ]:
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

In [ ]:
MODEL = "llama3.2"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [ ]:
# Quick health check. This should return content like b'Ollama is running'.
requests.get("http://localhost:11434", timeout=5).content

In [ ]:
def fetch_website_contents(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"
    }
    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    title = soup.title.string.strip() if soup.title and soup.title.string else "No title found"

    for tag in soup(["script", "style", "img", "input", "nav", "footer"]):
        tag.decompose()

    text = soup.get_text(separator="\n", strip=True)
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    cleaned_text = "\n".join(lines)

    return f"Title: {title}\n\nContents:\n{cleaned_text}"

In [ ]:
system_prompt = """
You are a practical assistant that analyzes website contents and writes concise summaries.
Ignore navigation, menus, repeated links, cookie banners, and footer clutter.
Respond in markdown without wrapping the answer in a code block.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of the website.
If the website includes news, announcements, products, services, or calls to action, summarize those too.

"""

In [ ]:
def messages_for(website_contents):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website_contents[:12_000]},
    ]

In [ ]:
def summarize(url):
    website_contents = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=messages_for(website_contents),
    )
    return response.choices[0].message.content

In [ ]:
def display_summary(url):
    display(Markdown(summarize(url)))

In [ ]:
display_summary("https://edwarddonner.com")

## Try Your Own Website

Change the URL below and rerun the cell. Simple static pages work best; JavaScript-heavy pages may scrape poorly with this basic `requests` + `BeautifulSoup` approach.

In [ ]:
display_summary("https://huggingface.co")